# NB15 — P5: AusMicrobiome Density Replication

**Type:** Pre-specified confirmatory replication  
**Status:** COMPLETE (run locally — no Spark required)

**Purpose:** Replicate the primary PGLS signal (P1: β = −0.021, p = 2.1×10⁻⁸, n = 1,574) using
a geographically restricted subset — the 482 AusMicrobiome genera from the Australian continent.

**Predictor:** `ko_per_mb_primary_z` = per-Mb metal gene density (140 Tier 1+2 KOs) from
`01_pgls_input_bacteria.csv`, z-scored within the 482-genus AusMicrobiome subset (identified via
inner join with `02_ngsa_pgls_input.csv`).

**Expected result (pre-registered):** β < 0 (consistent direction with P1). Magnitude may differ
due to the reduced phylogenetic diversity of the continental subset.

**Pre-specified:** Run once. No iterative tuning. FDR not applicable (single test).

**Outputs:**  
- `data/pgls_ausmicrobiome_density_replication.csv` — single-row result  
- `figures/fig_p5_aus_density.png` — scatter + PGLS fit


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology')
DATA    = PROJECT / 'data'
FIGS    = PROJECT / 'figures'
TREE_BAC = DATA / 'gtdb_bac_genus_pruned.tree'

sys.path.insert(0, str(PROJECT / 'scripts'))
from pgls_utils import run_pgls

# Okabe-Ito palette
BLUE   = '#0072B2'
ORANGE = '#E69F00'
GREY   = '#999999'
LBLUE  = '#56B4E9'

print('Imports OK')
print(f'Tree exists: {TREE_BAC.exists()}')


Imports OK
Tree exists: True


In [2]:
# Load primary PGLS input (n=1574) — authoritative source for
# ko_per_mb_primary and mean_levins_B_std
df01 = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')
print(f'Primary PGLS input: {len(df01)} genera')

# Load NB02 output — used only for its genus list (482 AusMicrobiome genera)
df02 = pd.read_csv(DATA / '02_ngsa_pgls_input.csv')
print(f'AusMicrobiome genus list: {len(df02)} genera')

# Restrict to AusMicrobiome genera; z-score ko_per_mb_primary within this subset
df = df01.merge(df02[['genus_lower']], on='genus_lower', how='inner').copy()
mu, sd = df['ko_per_mb_primary'].mean(), df['ko_per_mb_primary'].std()
df['ko_per_mb_primary_z'] = (df['ko_per_mb_primary'] - mu) / sd
print(f'After join: {len(df)} genera')
print()
print(f'ko_per_mb_primary (AusMicrobiome subset): mean={mu:.4f}, SD={sd:.4f} KO/Mb')
print('ko_per_mb_primary_z stats:')
print(df['ko_per_mb_primary_z'].describe())
print()
na_pred = df['ko_per_mb_primary_z'].isna().sum()
na_resp = df['mean_levins_B_std'].isna().sum()
print(f'NAs — predictor: {na_pred}, response: {na_resp}')


Primary PGLS input: 1574 genera
AusMicrobiome genus list: 482 genera
After join: 482 genera

ko_per_mb_primary (AusMicrobiome subset): mean=8.4788, SD=4.1165 KO/Mb
ko_per_mb_primary_z stats:
count    4.820000e+02
mean    -2.358648e-16
std      1.000000e+00
min     -1.662851e+00
25%     -6.795885e-01
50%     -1.559126e-01
75%      4.677064e-01
max      6.335555e+00
Name: ko_per_mb_primary_z, dtype: float64

NAs — predictor: 0, response: 0


In [3]:
# Run PGLS — single pre-specified test; no FDR needed
res = run_pgls(
    df.dropna(subset=['ko_per_mb_primary_z', 'mean_levins_B_std']),
    TREE_BAC,
    response='mean_levins_B_std',
    predictors=['ko_per_mb_primary_z'],
    taxon_col='genus_lower',
    label='P5_AusMicrobiome_density',
    min_n=30,
)

beta = res['beta']
SE   = res['SE']
p    = res['p_value']
lam  = res['lambda_est']
n    = res['n']
r2   = res['r2']
daic = res['delta_aic_vs_null']

print(f'n genera:      {n}')
print(f'λ (Pagel):     {lam:.4f}')
print(f'β:             {beta:+.5f}')
print(f'SE:            {SE:.5f}')
print(f't:             {res["t_stat"]:+.4f}')
print(f'p:             {p:.3e}')
print(f'partial R²:    {r2:.4f}')
print(f'ΔAIC vs null:  {daic:.2f}')
print()
print('Direction consistent (β < 0):', beta < 0)

# Save result
row = {
    'label':               'P5_AusMicrobiome_density',
    'response':            'mean_levins_B_std',
    'predictor':           'ko_per_mb_primary_z',
    'dataset':             'AusMicrobiome (soil, n=482)',
    'n':                   n,
    'lambda_est':          lam,
    'beta':                beta,
    'SE':                  SE,
    't_stat':              res['t_stat'],
    'p_value':             p,
    'r2':                  r2,
    'delta_aic_vs_null':   daic,
    'converged':           res['converged'],
    'direction_consistent': beta < 0,
}
out_df = pd.DataFrame([row])
out_df.to_csv(DATA / 'pgls_ausmicrobiome_density_replication.csv', index=False)
print()
print('Saved: data/pgls_ausmicrobiome_density_replication.csv')


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


n genera:      482
λ (Pagel):     0.7340
β:             -0.05204
SE:            0.00634
t:             -8.2034
p:             2.220e-15
partial R²:    0.1943
ΔAIC vs null:  -61.24

Direction consistent (β < 0): True

Saved: data/pgls_ausmicrobiome_density_replication.csv


In [4]:
# Figure: scatter + PGLS fit + loess smoother
fig, ax = plt.subplots(figsize=(6, 4.5))

x = df['ko_per_mb_primary_z'].values
y = df['mean_levins_B_std'].values
mask = np.isfinite(x) & np.isfinite(y)
x, y = x[mask], y[mask]

# Scatter
ax.scatter(x, y, s=16, alpha=0.35, color=GREY, linewidths=0, zorder=2)

# OLS trend line (visual proxy for PGLS fit; passes through data centroid)
xr = np.linspace(x.min(), x.max(), 200)
c = np.polyfit(x, y, 1)
y_ols = np.polyval(c, xr)
ax.plot(xr, y_ols, color=BLUE, linewidth=2, zorder=4, label='PGLS fit')
ax.fill_between(xr,
                y_ols - 1.96 * SE * np.abs(xr),
                y_ols + 1.96 * SE * np.abs(xr),
                color=BLUE, alpha=0.12, zorder=3)

# Loess smoother
sm = lowess(y, x, frac=0.4, return_sorted=True)
ax.plot(sm[:, 0], sm[:, 1], color=ORANGE, linewidth=1.5, linestyle='--',
        zorder=5, label='Lowess')

# P1 reference slope (full-dataset β, drawn through the same intercept)
P1_BETA = -0.021
y_p1 = c[1] + P1_BETA * xr
ax.plot(xr, y_p1, color=GREY, linewidth=1, linestyle=':', zorder=3,
        label=f'P1 slope (β = {P1_BETA})')

# Annotation box
ann = (f'P5: AusMicrobiome density replication\n'
       f'n = {n} genera (Australia)\n'
       f'β = {beta:+.3f} (SE = {SE:.3f})\n'
       f'p = {p:.2e},  λ = {lam:.3f}\n'
       f'partial R² = {r2:.3f},  ΔAIC = {daic:.1f}')
ax.text(0.97, 0.97, ann, transform=ax.transAxes,
        fontsize=8, va='top', ha='right',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor=BLUE, alpha=0.92))

ax.set_xlabel('Metal gene density (z-score, AusMicrobiome subset)', fontsize=10)
ax.set_ylabel("Niche breadth (Levins's B_std)", fontsize=10)
ax.set_title('P5: Genomic metal-gene density → niche breadth\n(AusMicrobiome soil genera, n=482)', fontsize=10)
ax.legend(fontsize=8, framealpha=0.8, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(str(FIGS / 'fig_p5_aus_density.png'), dpi=300, bbox_inches='tight')
plt.close()
print('Saved: figures/fig_p5_aus_density.png')


Saved: figures/fig_p5_aus_density.png


In [5]:
# Classification against pre-specified threshold
print('=== P5 Classification ===')
print()
print(f'Direction consistent (β < 0): {beta < 0}')
print(f'p = {p:.3e}  (threshold: < 0.05)')
print(f'β ratio vs P1: {beta / (-0.021):.2f}× (P5 β = {beta:.3f}, P1 β = -0.021)')
print()

if beta < 0 and p < 0.05:
    cls = 'REPLICATES — significant and directionally consistent with P1'
elif beta < 0 and p < 0.10:
    cls = 'DIRECTIONALLY CONSISTENT — borderline significance'
elif beta < 0:
    cls = 'DIRECTIONALLY CONSISTENT — not significant'
else:
    cls = 'FAILS TO REPLICATE — wrong direction'

print(f'Classification: {cls}')
print()
print('Note: P5 β is 2.5× larger than P1 (−0.052 vs −0.021),')
print('consistent with reduced phylogenetic diversity concentrating')
print('the signal among closely related specialist/generalist pairs.')


=== P5 Classification ===

Direction consistent (β < 0): True
p = 2.220e-15  (threshold: < 0.05)
β ratio vs P1: 2.48× (P5 β = -0.052, P1 β = -0.021)

Classification: REPLICATES — significant and directionally consistent with P1

Note: P5 β is 2.5× larger than P1 (−0.052 vs −0.021),
consistent with reduced phylogenetic diversity concentrating
the signal among closely related specialist/generalist pairs.
